# Week 5 Lab: Build an AMU Question-Answering Assistant

**A student has lost their card. Can we give them the correct procedure and show where it comes from?**

Collect three official AMU webpages, make their text searchable, and use the retrieved passages to answer student questions. Then ask a second LLM call to review the answer and compare the result with web search.

The core is Sections 0–6: one question, one collection, and four answer workflows. Allow roughly 75–90 minutes, including reading the results. Section 7 is an open evaluation exercise; the extensions retain keyword/hybrid retrieval and a second-model comparison.

Work individually; discussion and tools are welcome. Save private outputs under `local/student/2026-2027/week_5/`.

## 0) Setup

Select the same Python environment as in Weeks 3–4, for example `venvLLMDS`. `%pip` installs into the notebook's active kernel. Run the installation cell if packages are missing; restart the kernel if prompted.

Collection and retrieval run locally. For the answer and review calls, use `gemini-3.5-flash-lite` with a **Free Tier project**:

1. Open [Google AI Studio's API keys page](https://aistudio.google.com/api-keys). Choose a project whose plan is **Free Tier** and create its key. If your existing project is paid, use a separate free project for this lab.
2. Save that key as `GEMINI_API_KEY` in the repository's local `.env`.
3. Leave billing disabled. The notebook cannot identify your billing plan from the key; using a paid project's key can incur charges.

The core uses 4–10 model requests, depending on how many revisions the reviewer requests. Reruns and exercises use additional quota. If you receive `429 RESOURCE_EXHAUSTED`, check your project's [rate limits](https://aistudio.google.com/usage?timeRange=last-28-days&tab=rate-limit), wait for the indicated reset and rerun the affected cell. Paying is not part of this lab.

Section 5 uses the free **AI Studio browser interface** for the web-search comparison. The notebook makes no search-tool API calls. Availability and quotas depend on the account; see [pricing](https://ai.google.dev/gemini-api/docs/pricing) and [billing](https://ai.google.dev/gemini-api/docs/billing), checked 21 October 2026.

In [ ]:
%pip install requests beautifulsoup4 sentence-transformers rank-bm25 numpy pandas google-genai python-dotenv

In [ ]:
from pathlib import Path
import importlib
import json
import os
import re
import sys
import time

import numpy as np
import pandas as pd
import requests
from IPython.display import display, Markdown

course_root = next(
    folder for folder in [Path.cwd(), *Path.cwd().parents]
    if (folder / 'labs/week5_helpers.py').is_file()
)
examples = course_root / 'slides/examples/week5_retrieval'
sys.path.insert(0, str(course_root / 'labs'))
sys.path.insert(0, str(examples))
import week5_helpers
import workflow
importlib.reload(week5_helpers)
importlib.reload(workflow)
from week5_helpers import (
    load_or_download, make_chunks, build_context, retrieval_metrics,
    record_call, run_experiment, comparison_table,
)
from workflow import check_review

work = course_root / 'local/student/2026-2027/week_5'
work.mkdir(parents=True, exist_ok=True)
usage_log, results = [], {}  # Rerun setup to start a new experiment record.
print('Private outputs:', work)

## 1) Collect the official pages

The lecture began with six **fictional** documents. We now replace them with actual AMU pages. The IUT has its own administrative instructions, so we specify that the student attends the IUT.

Read the linked pages before asking a model. Locate the replacement instructions and note which facts an answer must include. The Wi-Fi and library pages give us other services to search later.

In [ ]:
sources = json.loads((examples / 'amu_sources.json').read_text())['sources']
display(pd.DataFrame(sources)[['title', 'scope', 'url']])
question = (
    "J'étudie à l'IUT d'Aix-Marseille. J'ai perdu ma carte étudiante. "
    "Comment la remplacer ?"
)
print('Question:', question)

`load_or_download` downloads each page with `requests`, then uses `BeautifulSoup` to extract headings and paragraphs from its main content. Its implementation is in [week5_helpers.py](week5_helpers.py).

It waits up to 5 seconds to connect and 30 seconds between received data. If the website times out, rejects the request or returns unusable HTML, it uses the **course snapshot from 21 October 2026** and prints which source failed. These are saved AMU pages, not generated answers. The table shows whether each document came from the website or the course snapshot.

HTML, source URL, capture date and checksum are saved locally. Later runs reuse the same files, including any fallback snapshot. To try downloading newer pages, choose a new `snapshot_dir` below. A course snapshot may differ from the current website; keep that difference in mind when comparing with web search.

In [ ]:
snapshot_dir = work / 'pages'
started = time.perf_counter()
with requests.Session() as session:
    documents = [load_or_download(source, snapshot_dir, session,
                                  fallback_dir=examples / 'snapshots') for source in sources]
collection_seconds = time.perf_counter() - started

display(pd.DataFrame([
    {'source': d['id'], 'paragraphs': len(d['sections']),
     'downloaded_at': d['downloaded_at'], 'origin': d['data_origin'],
     'reused_file': d['from_cache']}
    for d in documents
]))
print(f'Collection/extraction time this run: {collection_seconds:.2f} seconds')

In [ ]:
# Inspect the IUT text before turning it into model input.
iut = next(d for d in documents if d['id'] == 'AMU-IUT')
for section in iut['sections']:
    if 'carte' in section['heading'].lower():
        print(section['heading'])
        print(section['text'], '\n')

**Check the extraction:** compare the displayed text with the original page. Is the procedure complete? Are the fee and exceptions attached to the right heading? Menus, repeated navigation and missing paragraphs can all affect the answer before an LLM is called.

### Split the text into passages

A **chunk** is a passage searched separately. We group adjacent paragraphs under the same heading and split long sections into pieces of at most 80 words. Each chunk keeps its heading, institutional scope, source URL and download date. The ID stays the same while we use the same snapshot and splitting rule.

In [ ]:
chunks = make_chunks(documents, max_words=80)
(work / 'chunks.json').write_text(json.dumps(chunks, ensure_ascii=False, indent=2))
display(pd.DataFrame(chunks)[['chunk_id', 'heading', 'scope', 'url']])
print('Number of searchable passages:', len(chunks))
print(json.dumps(chunks[0], ensure_ascii=False, indent=2))

## 2) Retrieve passages for the full question

We reuse Week 4's multilingual sentence-embedding model. A vector represents each passage; the question gets a vector from the same model. Similar vectors receive higher scores.

The model may download weights on the first run. We use CPU here because the collection is small. Its input limit is set to 256 tokens to accommodate the short French passages and their headings.

In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model_name = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
embedder = SentenceTransformer(embedding_model_name, device='cpu')
embedder.max_seq_length = 256
texts = [c['heading'] + '\n' + c['text'] for c in chunks]
started = time.perf_counter()
emb_matrix = embedder.encode(texts, convert_to_numpy=True, normalize_embeddings=True)
index_seconds = time.perf_counter() - started
np.save(work / 'passage_vectors.npy', emb_matrix)
print('Embedding matrix:', emb_matrix.shape)  # One row per passage.
print(f'Embedding time: {index_seconds:.2f} seconds')
lengths = [len(embedder.tokenizer.encode(text)) for text in texts]
print('Passages exceeding the embedding input limit:', sum(n > 256 for n in lengths))

In [ ]:
def retrieve(question, k=4):
    query_vector = embedder.encode(
        [question], convert_to_numpy=True, normalize_embeddings=True,
    )[0]  # One question, so take its only vector.
    scores = emb_matrix @ query_vector  # Cosine similarity for normalized vectors.
    indices = np.argsort(-scores, kind='stable')[:k]
    return [dict(chunks[i], score=float(scores[i])) for i in indices]

selected = retrieve(question)
print('Full question sent to retrieval:', question)
for passage in selected:
    print(f"[{passage['chunk_id']}] similarity={passage['score']:.3f}")
    print(passage['heading'])
    print(passage['text'])
    print(passage['url'], '\n')

**Inspect before generating:** which returned passages answer the student's question? Did we miss an instruction? The similarity score ranks passages; it is not a probability that an answer will be correct.

From the full chunk table, identify **all** passages needed for this question. Enter their IDs below to compute retrieval precision and recall. This reference set comes from your reading of the sources, not from the retriever's top results.

In [ ]:
relevant_ids = set()  # Exercise: enter the relevant chunk IDs after reading the sources.
if relevant_ids:
    unknown = relevant_ids - {c['chunk_id'] for c in chunks}
    if unknown:
        raise ValueError(f'Unknown chunk IDs: {unknown}')
    print(retrieval_metrics([c['chunk_id'] for c in selected], relevant_ids))
else:
    print('Read the source passages, enter their IDs above, then rerun this cell.')

## 3) Answer from all documents, then from retrieved passages

We compare two inputs to the **same model with the same answer instructions**:

- **Full context:** all collected passages.
- **RAG:** the four passages returned for the full question.

The system instruction specifies how to answer. The user message contains the question and evidence. We give each passage its existing ID and URL so citations remain traceable.

Run the API setup below before continuing. It reads the repository `.env` without displaying the key. No Docker or generated-code execution is needed in this lab.

In [ ]:
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv(course_root / '.env', override=True)
api_key = os.getenv('GEMINI_API_KEY')
if not api_key:
    raise ValueError('Add GEMINI_API_KEY to the repository .env, then rerun this cell.')
model_name = 'gemini-3.5-flash-lite'
client = genai.Client(
    api_key=api_key,
    http_options=types.HttpOptions(
        timeout=120000, retry_options=types.HttpRetryOptions(attempts=1),
    ),
)
print('Hosted model:', model_name)

answer_instructions = (
    'Answer the student in French, concisely and with practical instructions. '
    'Use only supplied or retrieved official AMU evidence. '
    'Respect the faculty/service scope. Cite sources beside the facts they support. '
    'State when evidence is missing. Do not invent a procedure, price or deadline.'
)
base_config = types.GenerateContentConfig(
    system_instruction=answer_instructions,
    thinking_config=types.ThinkingConfig(thinking_level='low'),
    max_output_tokens=3000,
)

The next function makes the API call and records time and reported token usage, including failures. `finally` runs whether the call succeeds or raises an error. Raw responses and usage records stay in the private output directory.

A request is attempted once. If a timeout, quota or permission error occurs, inspect it before rerunning; rerunning sends a new request.

In [ ]:
def call_model(prompt, config, role, model=model_name):
    response, error = None, None
    started = time.perf_counter()
    try:
        response = client.models.generate_content(model=model, contents=prompt, config=config)
        if not response.text:
            raise ValueError('The model returned no answer text; inspect the saved response.')
        return response
    except Exception as exc:
        error = f'{type(exc).__name__}: {exc}'
        raise
    finally:
        row = record_call(usage_log, response, model, role,
                          time.perf_counter() - started, error)
        record = {'prompt': prompt, 'usage': row,
                  'response': response.model_dump(mode='json', exclude_none=True) if response else None}
        call_dir = work / 'calls'
        call_dir.mkdir(exist_ok=True)
        (call_dir / f'{time.time_ns()}.json').write_text(
            json.dumps(record, ensure_ascii=False, indent=2))

In [ ]:
def answer_prompt(question, passages, previous_answer='', feedback=''):
    return json.dumps({
        'question': question, 'evidence': build_context(passages),
        'previous_answer': previous_answer, 'review_feedback': feedback,
    }, ensure_ascii=False)


def write_answer(question, passages, previous_answer='', feedback=''):
    prompt = answer_prompt(question, passages, previous_answer, feedback)
    return call_model(prompt, base_config, role='answer').text

print(answer_prompt(question, selected))

Read the displayed prompt: it contains the full question and actual passage text, not just filenames or links.

The next cell sends **two requests**. `run_experiment` stores each result, elapsed time and all calls made by that workflow. It reports failures without replacing them with a worked example. Rerunning replaces that workflow's comparison row; the separate saved run files retain the earlier attempts.

In [ ]:
def rag_once():
    passages = retrieve(question)
    return {'answer': write_answer(question, passages), 'passages': passages}

full_result = run_experiment(
    'Full context', lambda: {'answer': write_answer(question, chunks), 'passages': chunks},
    usage_log, results, work,
)
rag_result = run_experiment('RAG', rag_once, usage_log, results, work)

**Check both answers against the IUT page.** Are they complete? Do their citations identify the passages that support each instruction? If both are correct, record that result; we do not need to manufacture an error to justify the next experiment.

## 4) Add the answer–review loop

A second LLM call receives the **question, retrieved passages and current answer**. It returns:

- `accepted`: the answer is complete and supported;
- `revise`: the passages contain the necessary information, but the answer needs a specified correction;
- `missing_information`: the supplied passages cannot answer the question.

For `revise`, send the previous answer and feedback to the writer. Keep the retrieved passages fixed. Stop on the other two statuses or after four attempts. Both roles use the same model in separate calls; four attempts mean at most eight calls.

The reviewer is part of the workflow being tested. You still check the final answer against the sources.

In [ ]:
review_instructions = (
    'Review the answer against the question and the supplied evidence. '
    'Check completeness, factual support, institutional scope and citations. '
    'Return JSON with exactly status and feedback. '
    'Use accepted if complete and supported; revise if specific corrections can be '
    'made from these passages; missing_information if these passages are insufficient. '
    'In feedback explain the decision and any required correction or missing fact. '
    'Do not ask for facts that the question does not require.'
)
review_config = base_config.model_copy(update={
    'system_instruction': review_instructions,
    'response_mime_type': 'application/json',
})


def review_answer(question, passages, answer):
    prompt = json.dumps({'question': question, 'evidence': build_context(passages),
                         'answer': answer}, ensure_ascii=False)
    response = call_model(prompt, review_config, role='review')
    review = json.loads(response.text)
    check_review(review)  # Validate status and feedback fields, not factual correctness.
    return review

In [ ]:
def answer_with_review():
    passages = retrieve(question)
    answer, feedback = '', ''
    trace = []
    status = 'attempt_limit'
    for attempt in range(4):
        answer = write_answer(question, passages, answer, feedback)
        review = review_answer(question, passages, answer)
        trace.append({'attempt': attempt + 1, 'answer': answer, 'review': review})
        if review['status'] != 'revise':
            status = review['status']
            break
        feedback = review['feedback']
    return {'status': status, 'answer': answer, 'trace': trace, 'passages': passages}

review_result = run_experiment('RAG + review', answer_with_review, usage_log, results, work)
for step in review_result.get('trace', []):
    print('Attempt', step['attempt'])
    print(step['answer'])
    print('Review:', step['review'], '\n')

**Inspect the actual trace.** Did the reviewer accept the first answer or ask for a change? If it requested a change, did the next answer address that feedback? Were any corrections unjustified?

`missing_information` means these passages are insufficient; another page might still contain the information. `attempt_limit` means the last answer remains unresolved. Neither status counts as a verified answer.

## 5) Compare with web search in AI Studio

Keep the same question and answer requirements. This time, let the model search online instead of supplying our saved pages.

1. Open a new chat in [Google AI Studio](https://aistudio.google.com/). Use its free access, without linking a paid API key.
2. Select `gemini-3.5-flash-lite` and enable **Grounding with Google Search** in the tools/settings panel.
3. Copy the prompt printed below, run it once, and inspect the answer's sources.

Writing “search the web” alone does not enable search. Use the search setting and check the returned source links. If free search is unavailable for your account, record that limitation and continue with the three notebook workflows; do not enable billing.

We use the browser here because search through this model's API requires paid-tier access. [Google documents the AI Studio alternative](https://ai.google.dev/gemini-api/docs/pricing).

In [ ]:
web_prompt = (
    answer_instructions + '\n\n' + question
    + '\nSearch official AMU webpages and cite the supporting sources.'
)
(work / 'web_search_prompt.txt').write_text(web_prompt)
print(web_prompt)  # Copy into AI Studio; this cell makes no model request.

Save the browser answer and its source links under the private output directory printed in Section 0. Record the model, date and whether search was enabled. Open the cited pages: did search find additional information, use a different faculty's procedure, or see a newer page than our snapshot?

Compare this answer's **quality and sources** with the notebook results. The browser may use different generation settings; this is a comparison of complete systems. Do not invent API token counts or search costs for the browser result. If you time it, measure from clicking Run until completion and record that duration separately from the notebook's timings.

## 6) Compare quality, time and token usage

The table contains the three **notebook workflows**: full context, RAG, and RAG with review. It uses their latest runs, includes every writer/reviewer call, and keeps failures visible. Elapsed time covers retrieval and model calls; collection and embedding are reported separately. Compare the saved browser answer alongside these results when assessing quality.

On a Free Tier project, these model calls use free quota. `paid_rate_estimate_usd` answers a different question: what would the reported tokens cost at standard paid rates? It is not a bill. Unknown token counts remain missing.

For the course model, the reference rates are $0.30 per million input tokens and $2.50 per million output/thinking tokens, checked on 21 October 2026. See [Google's pricing](https://ai.google.dev/gemini-api/docs/pricing).

In [ ]:
report = comparison_table(results).rename(columns={'cost_estimate_usd': 'paid_rate_estimate_usd'})
report = report.drop(columns=['reported_search_queries'])
display(report)
print(f'Collection/extraction: {collection_seconds:.2f} s; embeddings: {index_seconds:.2f} s')
report.to_csv(work / 'workflow_comparison.csv', index=False)
print('Saved:', work / 'workflow_comparison.csv')

**Make the quality comparison yourself before choosing a workflow.** For each actual answer, record:

1. Which required facts are correct or missing?
2. Which claims are supported by the cited source? Does its faculty/service match the student?
3. Was an unsupported statement added during review?
4. Did extra calls improve the answer enough to justify their time and token usage?

An accepted reviewer status is not the quality score. A failed request remains a failed result. One question is enough to inspect the mechanism, but not enough to declare a generally best method.

## 7) Exercise — Test the assistant on new questions

Use a Wi-Fi question as a **development case**. Read the source, define the expected facts, inspect retrieval and make one change if needed: for example, change the number of passages or the review instructions. Keep the original card question as a regression check.

Then freeze your settings. Write three **new test questions**, including a library question and one requiring information absent from our collection (for example, yesterday's number of replacement-card requests). Before calling the models, write the expected facts or expected statement of missing information, with the supporting URLs.

Implement the comparison on these test questions. Keep the wording identical across workflows. Count correct, sufficiently complete answers and supported citations; retain failed requests. Compare time and token usage for the notebook workflows. Compare the browser answers by quality and sources; keep any manually measured browser time separate. Explain differences in source coverage or freshness. Stay within your free quota: run one question at a time and resume after a quota reset if needed. Save your notes locally.

If you add a page, rebuild **chunks, text lists, embeddings and any keyword index** before using it. Adding a document to a Python list alone does not update retrieval.

In [ ]:
# Your test questions, reference facts, evaluation code and conclusion.

## Extensions after the core comparison

### A) Compare another model with web search

In a new **free AI Studio chat**, choose `gemini-3.8-flash`, enable Google Search and reuse the prompt below. Save the answer and citations beside the first browser result. Use this extension only if that model and search are available without linking a paid key.

Check the same required facts and source support. Changing the model can also change which searches it chooses; this compares complete systems.

In [ ]:
print('Second browser model: gemini-3.8-flash')
print(web_prompt)  # Same prompt; run it in AI Studio, not through the paid search API.

### B) Keep the original keyword and hybrid retrieval comparison

**BM25** scores passages by query-word matches, adjusted for word frequency and passage length. Use the full question, just as for embedding search.

A **hybrid** combines keyword and embedding results. Here we scale each method's scores to 0–1 within this query, then take a weighted average. `alpha=0` uses only BM25; `alpha=1` uses only embeddings. This is a simple baseline to test, not a calibrated probability or an assumed improvement.

In [ ]:
from rank_bm25 import BM25Okapi


def words(text):
    return re.findall(r"\w+", text.lower())

bm25 = BM25Okapi([words(text) for text in texts])


def rank_passages(scores, k):
    indices = np.argsort(-scores, kind='stable')[:k]
    return [dict(chunks[i], score=float(scores[i])) for i in indices]


def retrieve_bm25(question, k=4):
    return rank_passages(bm25.get_scores(words(question)), k)


def scale_scores(scores):
    span = scores.max() - scores.min()
    return (scores - scores.min()) / span if span else np.zeros_like(scores)


def retrieve_hybrid(question, k=4, alpha=0.5):
    keyword_scores = bm25.get_scores(words(question))
    vector = embedder.encode([question], convert_to_numpy=True, normalize_embeddings=True)[0]
    embedding_scores = emb_matrix @ vector
    combined = (1 - alpha) * scale_scores(keyword_scores) + alpha * scale_scores(embedding_scores)
    return rank_passages(combined, k)

for name, search in [('BM25', retrieve_bm25), ('Embeddings', retrieve), ('Hybrid', retrieve_hybrid)]:
    hits = search(question)
    print(name, [c['chunk_id'] for c in hits])
    if relevant_ids:
        print(retrieval_metrics([c['chunk_id'] for c in hits], relevant_ids))

Change `k` or `alpha` using your development questions. Inspect which relevant passages are gained or lost, then evaluate the chosen settings on the untouched test questions. If changing the retriever changes the answer, use the retrieved text to explain why.